# CineGraph: project demonstration

This notebook demonstrates the running API and reads saved evaluation reports.

**Before running:** start Docker Desktop, run `make docker-run`,
and populate the database with `make setup-data`.

Select the project's `.venv` Python kernel.
The optional AI cell makes one OpenAI API request.


In [1]:
from pathlib import Path
import json
import sys

import httpx
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "app").is_dir():
    raise RuntimeError("Open this notebook from the project root.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from app.config import Settings

BASE_URL = "http://localhost:8000"
settings = Settings()

def fetch(path, **params):
    response = httpx.get(
        BASE_URL + path, params=params, timeout=30
    )
    response.raise_for_status()
    return response.json()


## 1. Database connectivity

Check that FastAPI can reach Neo4j and the GDS plugin.


In [2]:
health = fetch("/health/database")
display(health)
assert health["status"] == "ok"


{'status': 'ok', 'database': 'neo4j', 'gds_version': '2.13.14'}

## 2. Personalized recommendations

Find unseen films for MovieLens user 1.
The score aggregates shared-like counts from supporting viewers.
It is not a probability.


In [3]:
recommendations = fetch(
    "/api/users/1/recommendations", limit=5
)
assert recommendations
display(pd.DataFrame(recommendations))


,movieId,title,supportingUsers,averageRating,score
0,589,Terminator 2: Judgment Day (1991),18,4.47,1206
1,2918,Ferris Bueller's Day Off (1986),16,4.38,1106
2,858,"Godfather, The (1972)",16,4.75,1095
3,2762,"Sixth Sense, The (1999)",17,4.50,1093
4,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),16,4.38,1069


## 3. Recommendation evidence

Inspect the first recommendation.
Supporting-user counts and shared-like totals should match
the corresponding recommendation fields.


In [4]:
first = recommendations[0]
evidence = fetch(
    f"/api/users/1/recommendations/{first['movieId']}/why"
)
display(pd.DataFrame(evidence))

assert len(evidence) == first["supportingUsers"]
assert sum(row["sharedLikes"] for row in evidence) == first["score"]
print("Evidence agrees with the recommendation.")


,supportingUserId,sharedLikes,sharedMovies,recommendedMovieRating
0,414,130,"[Toy Story (1995), Grumpier Old Men (1995), Se...",5.0
1,380,87,"[Toy Story (1995), Heat (1995), Seven (a.k.a. ...",5.0
2,597,75,"[Toy Story (1995), Seven (a.k.a. Se7en) (1995)...",5.0
3,202,67,"[Toy Story (1995), Heat (1995), Usual Suspects...",5.0
4,555,66,"[Toy Story (1995), Grumpier Old Men (1995), Us...",5.0
5,45,65,"[Toy Story (1995), Heat (1995), Usual Suspects...",5.0
6,469,64,"[Toy Story (1995), Seven (a.k.a. Se7en) (1995)...",4.0
7,474,63,"[Toy Story (1995), Seven (a.k.a. Se7en) (1995)...",4.0
8,57,62,"[Toy Story (1995), Usual Suspects, The (1995),...",4.0
9,249,62,"[Toy Story (1995), Seven (a.k.a. Se7en) (1995)...",4.0


Evidence agrees with the recommendation.


## 4. Similar movie audiences

Movie 1 is Toy Story. Similarity comes from positive-rating
audience overlap rather than plot descriptions.


In [5]:
similar = fetch("/api/movies/1/similar", limit=5)
display(pd.DataFrame(similar))
assert all(row["movieId"] != 1 for row in similar)


,movieId,title,commonLikers,similarity
0,3114,Toy Story 2 (1999),46,0.2930
1,260,Star Wars: Episode IV - A New Hope (1977),78,0.2889
2,2762,"Sixth Sense, The (1999)",60,0.2830
3,480,Jurassic Park (1993),64,0.2807
4,364,"Lion King, The (1994)",57,0.2780


## 5. Centrality and community discovery

PageRank measures historical network centrality.
Louvain communities group connected movie audiences;
their identifiers can change after rebuilding.


In [6]:
display(pd.DataFrame(
    fetch("/api/movies/trending", limit=5)
))
display(pd.DataFrame(
    fetch("/api/users/1/discover", limit=5)
))


,movieId,title,pageRank
0,6286,"Man Without a Past, The (Mies vailla menneisyy...",17.623337
1,3306,"Circus, The (1928)",12.716484
2,1913,Picnic at Hanging Rock (1975),10.948867
3,6890,Elephant (2003),10.273093
4,2066,Out of the Past (1947),9.770869


,movieId,title,communityId,likedInCommunity,exampleLikedMovies,pageRank
0,858,"Godfather, The (1972)",39,93,"[Toy Story (1995), Heat (1995), Seven (a.k.a. ...",5.354394
1,7153,"Lord of the Rings: The Return of the King, The...",39,93,"[Toy Story (1995), Heat (1995), Seven (a.k.a. ...",4.884536
2,589,Terminator 2: Judgment Day (1991),39,93,"[Toy Story (1995), Heat (1995), Seven (a.k.a. ...",3.883621
3,4993,"Lord of the Rings: The Fellowship of the Ring,...",39,93,"[Toy Story (1995), Heat (1995), Seven (a.k.a. ...",3.207012
4,2918,Ferris Bueller's Day Off (1986),39,93,"[Toy Story (1995), Heat (1995), Seven (a.k.a. ...",3.106758


## 6. API-key protection

A request without an application key must be rejected
before AI generation is invoked.


In [7]:
response = httpx.post(
    BASE_URL + "/api/ask",
    json={"question": "Recommend 5 movies for user 1"},
    timeout=10,
)
print("Status:", response.status_code)
display(response.json())
assert response.status_code == 401


Status: 401


{'detail': 'Missing or invalid API key.'}

## 7. Optional natural-language demonstration

Change `RUN_LIVE_AI` to `True` to make one paid API request.
The application key stays in memory and is not printed.

The response exposes the approved Cypher, its parameters,
and the movie facts retrieved from Neo4j.


In [8]:
RUN_LIVE_AI = False

if RUN_LIVE_AI:
    response = httpx.post(
        BASE_URL + "/api/ask",
        headers={
            "X-API-Key": settings.app_api_key.get_secret_value()
        },
        json={"question": "Recommend 5 movies for user 1"},
        timeout=45,
    )
    response.raise_for_status()
    answer = response.json()
    print("Approved query:", answer["queryName"])
    print("Parameters:", answer["parameters"])
    print(answer["cypher"])
    display(pd.DataFrame(answer["results"]))
else:
    print("Live AI demonstration skipped.")


Live AI demonstration skipped.


## 8. Link-prediction evaluation

These are saved results from `make ml`, not a new training run.

The partial feature graph excludes supervised training and test links.
Unrated negative samples represent unknown interest, not confirmed dislike.
The metrics describe sampled hidden-link recovery rather than
future recommendation accuracy.


In [9]:
report_path = ROOT / "reports" / "link_prediction_metrics.json"
ml_report = json.loads(report_path.read_text(encoding="utf-8"))

display(pd.DataFrame({
    name: ml_report[name]
    for name in ("popularity_baseline", "logistic_regression")
}).T)

display(ml_report["limitations"])


,roc_auc,average_precision
popularity_baseline,0.881555,0.884487
logistic_regression,0.899918,0.907315


['Unrated movies are not confirmed dislikes.',
 'Random edge split does not measure future behavior.',
 'Balanced sampled candidates do not represent the full movie catalog.',
 'This experiment does not replace the API recommender.']

## 9. Query benchmark

These are saved measurements from `make benchmark`.

The experiment compares a forced label scan and an index lookup
for a missing movie ID. It does not measure whole-app performance.


In [10]:
report_path = ROOT / "reports" / "query_benchmark.json"
benchmark = json.loads(report_path.read_text(encoding="utf-8"))

display(pd.DataFrame([
    {
        "method": name,
        "median_ms": result["median_elapsed_ms"],
        "database_hits": result["total_db_hits"],
    }
    for name, result in benchmark["measurements"].items()
]))

assert benchmark["identical_results"]
display(benchmark["limitations"])


,method,median_ms,database_hits
0,scan,4.317375,19485
1,index,1.042542,1


['Both queries run against the same already-indexed database.',
 'The scan hint simulates lookup without using that index.',
 'Warm-cache client timings include driver and network overhead.',
 'This small lookup does not measure whole-app performance.']

## Interpretation

- API results demonstrate graph-based recommendation and discovery.
- Supporting graph evidence explains the recommendation score.
- The AI selects guarded queries; movie results come from Neo4j.
- The ML model is evaluated separately from the API recommender.
- Benchmark results apply to the recorded lookup and environment.
